In [1]:
import os
import ast

import pandas as pd
import matplotlib.pyplot as plt
import scipy

from plotting_utils import (
    get_train_domain,
    get_test_domain,
    get_model_name,
    wanted_metrics
)

pd.set_option('display.float_format', lambda x: '%.2f' % x)
pd.set_option('display.multi_sparse', True)
PROJECT_DIR = os.environ["MDS"]

# Load and format results

In [2]:
results_df = pd.read_csv(
    #f"{PROJECT_DIR}/results_mayarr_sample/experiments_mayarr/aggregated_results.csv",
    f"{PROJECT_DIR}/results_mayarr/experiments_mayarr/aggregated_results.csv",
    header=[0,1]
)

results_df.columns = pd.Index([
  f"{c[0]}_{c[1]}" if c[1]!="nan" else c[0] for c in results_df.columns
])

results_df["train_domain"] = results_df.apply(get_train_domain, axis="columns")
results_df["test_domain"] = results_df.dataset.map(get_test_domain)
results_df["model_type"] = results_df.model.map(get_model_name)

# Remove validation set
results_df = results_df[results_df.split=="test"]

new_columns = [
    "model_type",
    "train_domain",
    "test_domain",
] + wanted_metrics

# Add placeholders
for c in new_columns:
    if c not in results_df.columns:
        results_df[c] = pd.NA

results_df = results_df[new_columns]

def format_entry(entry: str):
    try:
        entry = ast.literal_eval(entry)
    except ValueError as err:
        print(f"Could not convert '{entry}' as literal")
        return pd.NA
    return entry.get("mean", pd.NA)

# Display only the mean
for metric in wanted_metrics:
    results_df[metric] = results_df[metric].map(format_entry)

# Change compression so that higher score = more compressed summary
# 1 - (# sentences in summary / # sentences in source docs)
for m in ["compression_sentence", "compression_token"]:
    results_df[m] = 1 - results_df[m]


# Hack
results_df = results_df.fillna(-1)

Could not convert 'nan' as literal
Could not convert 'nan' as literal
Could not convert 'nan' as literal
Could not convert 'nan' as literal
Could not convert 'nan' as literal
Could not convert 'nan' as literal
Could not convert 'nan' as literal
Could not convert 'nan' as literal
Could not convert 'nan' as literal
Could not convert 'nan' as literal
Could not convert 'nan' as literal
Could not convert 'nan' as literal
Could not convert 'nan' as literal
Could not convert 'nan' as literal
Could not convert 'nan' as literal
Could not convert 'nan' as literal
Could not convert 'nan' as literal
Could not convert 'nan' as literal
Could not convert 'nan' as literal
Could not convert 'nan' as literal
Could not convert 'nan' as literal
Could not convert 'nan' as literal
Could not convert 'nan' as literal
Could not convert 'nan' as literal
Could not convert 'nan' as literal
Could not convert 'nan' as literal
Could not convert 'nan' as literal
Could not convert 'nan' as literal
Could not convert 'n

/tmp/ipykernel_2515689/1805173534.py:50: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  results_df = results_df.fillna(-1)


In [3]:
a = results_df.set_index("model_type").loc["PRIMERA"].set_index(["train_domain", "test_domain"]).sort_index()
a[["prometheus_factuality", "prometheus_consistency", "rouge_rouge1"]]


prometheus_factuality  prometheus_consistency  \
train_domain test_domain                                                  
Convo        Convo                         1.45                    1.58   
             News+                        -1.00                   -1.00   
             Science                       1.92                    1.98   
News+        Convo                         1.45                    1.52   
             News+                        -1.00                   -1.00   
             Science                       1.93                    1.96   
Science      Convo                         1.38                    1.53   
             News+                        -1.00                   -1.00   
             Science                       1.93                    1.97   

                          rouge_rouge1  
train_domain test_domain                
Convo        Convo                0.19  
             News+                0.38  
             Science              0.24  
News+        Convo                0.19  
             News+                0.38  
             Science              0.24  
Science      Convo                0.19  
             News+                0.38  
             Science              0.24

In [4]:
results_df.columns

Index(['model_type', 'train_domain', 'test_domain', 'unieval_factuality',
       'prometheus_factuality', 'prometheus_consistency', 'unieval_relevance',
       'prometheus_relevance', 'rouge_rouge1', 'rouge_rougeLsum', 'moverscore',
       'unieval_fluency', 'prometheus_fluency', 'unieval_coherence',
       'prometheus_coherence', 'compression_sentence', 'compression_token',
       'prometheus_coverage'],
      dtype='object')

In [5]:
results_df.groupby("model_type").agg({
    "compression_sentence": ["mean", "std"],
    "compression_token": ["mean", "std"]
})


compression_sentence      compression_token     
                                   mean  std              mean  std
model_type                                                         
ConvoSumm                          0.72 0.14              0.74 0.14
Llama 3.1                          0.64 0.11              0.55 0.05
Llama 3.1 (0-shot)                 0.64 0.12              0.55 0.06
PRIMERA                            0.66 0.06              0.56 0.05
SummN                              0.71 0.17              0.70 0.18
ground_truth                       0.68 0.25              0.67 0.23

In [6]:
a = results_df.set_index(["model_type", "train_domain", "test_domain"]).sort_index()
print(a.to_csv(float_format='%.2f'))
a

model_type,train_domain,test_domain,unieval_factuality,prometheus_factuality,prometheus_consistency,unieval_relevance,prometheus_relevance,rouge_rouge1,rouge_rougeLsum,moverscore,unieval_fluency,prometheus_fluency,unieval_coherence,prometheus_coherence,compression_sentence,compression_token,prometheus_coverage
ConvoSumm,Convo,Convo,0.64,2.21,2.25,0.70,2.44,0.31,0.21,0.06,0.91,2.67,0.60,2.49,0.91,0.91,-1
ConvoSumm,Convo,News+,0.76,-1.00,-1.00,0.51,-1.00,0.24,0.14,0.00,0.90,-1.00,0.58,-1.00,0.77,0.84,-1
ConvoSumm,Convo,Science,0.68,1.94,1.92,0.63,2.21,0.23,0.14,0.02,0.91,2.17,0.61,2.05,0.76,0.85,-1
ConvoSumm,News+,Convo,0.70,1.62,1.67,0.76,2.08,0.23,0.13,0.01,0.70,1.79,0.78,1.80,0.71,0.62,-1
ConvoSumm,News+,News+,0.78,-1.00,-1.00,0.79,-1.00,0.45,0.23,0.20,0.75,-1.00,0.82,-1.00,0.48,0.54,-1
ConvoSumm,News+,Science,0.71,1.82,-1.00,0.69,2.18,0.24,0.14,0.04,0.82,-1.00,0.69,-1.00,0.55,0.55,-1
ConvoSumm,Science,Convo,0.46,1.23,1.33,0.23,1.42,0.22,0.15,-0.02,0.58,1.32,0.25,1.24,0.87,0.85,-1
Con

unieval_factuality  \
model_type         train_domain test_domain                       
ConvoSumm          Convo        Convo                      0.64   
                                News+                      0.76   
                                Science                    0.68   
                   News+        Convo                      0.70   
                                News+                      0.78   
                                Science                    0.71   
                   Science      Convo                      0.46   
                                News+                      0.64   
                                Science                    0.60   
Llama 3.1          Convo        Convo                      0.83   
                                News+                      0.88   
                                Science                    0.88   
                   News+        Convo                      0.82   
                                News+                      0.88   
                                Science                    0.87   
                   Science      Convo                      0.83   
                                News+                      0.88   
                                Science                    0.88   
Llama 3.1 (0-shot) zero-shot    Convo                      0.82   
                                News+                      0.88   
                                Science                    0.88   
PRIMERA            Convo        Convo                      0.81   
                                News+                      0.81   
                                Science                    0.83   
                   News+        Convo                      0.81   
                                News+                      0.81   
                                Science                    0.82   
                   Science      Convo                      0.82   
                                News+                      0.81   
                                Science                    0.84   
SummN              Convo        Convo                      0.67   
                                News+                      0.90   
                                Science                    0.80   
                   News+        Convo                      0.67   
                                News+                      0.72   
                                Science                    0.64   
                   Science      Convo                      0.40   
                                News+                      0.64   
                                Science                    0.56   
ground_truth       Convo        Convo                      0.71   
                   News+        News+                      0.70   
                   Science      Science                    0.55   

                                             prometheus_factuality  \
model_type         train_domain test_domain                          
ConvoSumm          Convo        Convo                         2.21   
                                News+                        -1.00   
                                Science                       1.94   
                   News+        Convo                         1.62   
                                News+                        -1.00   
                                Science                       1.82   
                   Science      Convo                         1.23   
                                News+                        -1.00   
                                Science                       1.57   
Llama 3.1          Convo        Convo                         3.06   
                                News+                        -1.00   
                                Science                      -1.00   
                   News+        Convo                         3.06   
                                News+         

In [7]:
a = results_df[results_df.model_type=="PRIMERA"].set_index(["model_type", "train_domain", "test_domain"]).sort_index()
a.transpose()

model_type             PRIMERA                                            \
train_domain             Convo               News+               Science   
test_domain              Convo News+ Science Convo News+ Science   Convo   
unieval_factuality        0.81  0.81    0.83  0.81  0.81    0.82    0.82   
prometheus_factuality     1.45 -1.00    1.92  1.45 -1.00    1.93    1.38   
prometheus_consistency    1.58 -1.00    1.98  1.52 -1.00    1.96    1.53   
unieval_relevance         0.40  0.42    0.62  0.41  0.42    0.62    0.41   
prometheus_relevance      1.93 -1.00    2.38  1.98 -1.00    2.36    1.92   
rouge_rouge1              0.19  0.38    0.24  0.19  0.38    0.24    0.19   
rouge_rougeLsum           0.11  0.18    0.13  0.11  0.18    0.13    0.11   
moverscore               -0.01  0.13    0.06 -0.01  0.13    0.06   -0.01   
unieval_fluency           0.65  0.61    0.80  0.66  0.62    0.79    0.65   
prometheus_fluency        1.34 -1.00    1.83  1.29 -1.00    1.83    1.32   
unieval_coherence         0.68  0.69    0.73  0.72  0.69    0.74    0.71   
prometheus_coherence      1.30 -1.00    1.80  1.27 -1.00    1.80    1.27   
compression_sentence      0.73  0.61    0.62  0.75  0.62    0.63    0.74   
compression_token         0.59  0.52    0.53  0.63  0.52    0.53    0.62   
prometheus_coverage      -1.00 -1.00   -1.00 -1.00 -1.00   -1.00   -1.00   

model_type                            
train_domain                          
test_domain            News+ Science  
unieval_factuality      0.81    0.84  
prometheus_factuality  -1.00    1.93  
prometheus_consistency -1.00    1.97  
unieval_relevance       0.42    0.62  
prometheus_relevance   -1.00    2.36  
rouge_rouge1            0.38    0.24  
rouge_rougeLsum         0.18    0.13  
moverscore              0.13    0.06  
unieval_fluency         0.61    0.80  
prometheus_fluency     -1.00    1.82  
unieval_coherence       0.70    0.74  
prometheus_coherence   -1.00    1.79  
compression_sentence    0.62    0.62  
compression_token       0.52    0.53  
prometheus_coverage    -1.00   -1.00

# Q: How well is each model trained _on its domain_?


In [8]:
# Limit to models where the model is trained and tested on the same domain
same_domain_analysis = results_df[
    results_df.apply(lambda x: x.train_domain==x.test_domain, axis="columns")
].sort_values(by=["train_domain", "test_domain"])
same_domain_analysis = same_domain_analysis.loc[:, ["train_domain", "test_domain", "model_type"]+wanted_metrics]

same_domain_performance_dict = same_domain_analysis.set_index(["train_domain", "model_type"])[wanted_metrics].to_dict(orient="index")
same_domain_performance_dict

{('Convo', 'ground_truth'): {'unieval_factuality': 0.7127423688060528,
  'prometheus_factuality': 3.032,
  'prometheus_consistency': 3.128,
  'unieval_relevance': 0.9855425109640278,
  'prometheus_relevance': 3.256,
  'rouge_rouge1': -1.0,
  'rouge_rougeLsum': -1.0,
  'moverscore': 0.9999999998240994,
  'unieval_fluency': 0.8962882354893531,
  'prometheus_fluency': 3.276,
  'unieval_coherence': 0.729176020702826,
  'prometheus_coherence': 3.484,
  'compression_sentence': 0.8975773848277229,
  'compression_token': 0.8507181864684161,
  'prometheus_coverage': -1},
 ('Convo', 'PRIMERA'): {'unieval_factuality': 0.8141133978258701,
  'prometheus_factuality': 1.448,
  'prometheus_consistency': 1.58,
  'unieval_relevance': 0.4026709201737799,
  'prometheus_relevance': 1.932,
  'rouge_rouge1': 0.19090370176435598,
  'rouge_rougeLsum': 0.10950630435403477,
  'moverscore': -0.011266764867552136,
  'unieval_fluency': 0.653483834363345,
  'prometheus_fluency': 1.344,
  'unieval_coherence': 0.67855

In [9]:
a = same_domain_analysis.set_index(["model_type"]).sort_index()
print(a.to_csv())
a

model_type,train_domain,test_domain,unieval_factuality,prometheus_factuality,prometheus_consistency,unieval_relevance,prometheus_relevance,rouge_rouge1,rouge_rougeLsum,moverscore,unieval_fluency,prometheus_fluency,unieval_coherence,prometheus_coherence,compression_sentence,compression_token,prometheus_coverage
ConvoSumm,Convo,Convo,0.6448624135535097,2.208,2.248,0.6982185812416573,2.44,0.3087363026153122,0.21095126798276165,0.0556925312999886,0.9145819456531356,2.672,0.5996400811017836,2.488,0.9137200475966535,0.9062447934029834,-1
ConvoSumm,News+,News+,0.7753847037887037,-1.0,-1.0,0.794853197114234,-1.0,0.4533680919961548,0.22964031636124188,0.19757044789092731,0.7497246372593714,-1.0,0.8247894365782651,-1.0,0.48125052299165694,0.5405493341023895,-1
ConvoSumm,Science,Science,0.6019692657086441,1.5718043135192004,1.5728563913729616,0.4949875539961723,1.842714360862704,0.276839807129176,0.1628495558898957,0.03755076071382911,0.6915942894546336,1.4726315789473685,0.3776514048580986,1.467

,train_domain,test_domain,unieval_factuality,prometheus_factuality,prometheus_consistency,unieval_relevance,prometheus_relevance,rouge_rouge1,rouge_rougeLsum,moverscore,unieval_fluency,prometheus_fluency,unieval_coherence,prometheus_coherence,compression_sentence,compression_token,prometheus_coverage
model_type,,,,,,,,,,,,,,,,,
ConvoSumm,Convo,Convo,0.64,2.21,2.25,0.70,2.44,0.31,0.21,0.06,0.91,2.67,0.60,2.49,0.91,0.91,-1
ConvoSumm,News+,News+,0.78,-1.00,-1.00,0.79,-1.00,0.45,0.23,0.20,0.75,-1.00,0.82,-1.00,0.48,0.54,-1
ConvoSumm,Science,Science,0.60,1.57,1.57,0.49,1.84,0.28,0.16,0.04,0.69,1.47,0.38,1.47,0.72,0.79,-1
Llama 3.1,Convo,Convo,0.83,3.06,3.21,0.93,3.35,0.22,0.15,0.05,0.93,4.00,0.95,3.98,0.75,0.52,-1
Llama 3.1,News+,News+,0.88,-1.00,-1.00,0.92,-1.00,0.40,0.21,0.17,0.95,-1.00,0.94,-1.00,0.66,0.62,-1
Llama 3.1,Science,Science,0.88,-1.00,-1.00,0.91,-1.00,0.25,0.16,0.09,0.93,-1.00,0.97,-1.00,0.52,0.51,-1
PRIMERA,Convo,Convo,0.81,1.45,1.58,0.40,1.93,0.19,0.11,-0.01,0.65,1.34,0.68,1.30,0.73,0.59,-1
PRIMERA,News+,News+,0.81,-1.00,-1.00,0.42,-1.00,0.38,0.18,0.13,0.62,-1.00,0.69,-1.00,0.62,0.52,-1
PRIMERA,Science,Science,0.84,1.93,1.97,0.62,2.36,0.24,0.13,0.06,0.80,1.82,0.74,1.79,0.62,0.53,-1


In [10]:
a = same_domain_analysis.groupby(["model_type"]).mean(numeric_only=True).to_csv(float_format='%.2f')
print(a)

model_type,unieval_factuality,prometheus_factuality,prometheus_consistency,unieval_relevance,prometheus_relevance,rouge_rouge1,rouge_rougeLsum,moverscore,unieval_fluency,prometheus_fluency,unieval_coherence,prometheus_coherence,compression_sentence,compression_token,prometheus_coverage
ConvoSumm,0.67,0.93,0.94,0.66,1.09,0.35,0.20,0.10,0.79,1.05,0.60,0.99,0.71,0.74,-1.00
Llama 3.1,0.86,0.35,0.40,0.92,0.45,0.29,0.17,0.10,0.94,0.67,0.95,0.66,0.64,0.55,-1.00
PRIMERA,0.82,0.79,0.85,0.48,1.10,0.27,0.14,0.06,0.69,0.72,0.70,0.70,0.66,0.55,-1.00
SummN,0.65,0.97,0.98,0.66,1.14,0.35,0.20,0.10,0.76,1.10,0.63,1.06,0.69,0.70,-1.00
ground_truth,0.66,1.28,1.31,0.99,1.67,-1.00,-1.00,1.00,0.77,1.28,0.64,1.33,0.68,0.67,-1.00



In [11]:
a = same_domain_analysis[same_domain_analysis.model_type!='ground_truth'].groupby(["train_domain"]).mean(numeric_only=True)
print(a.to_csv(float_format='%.2f'))
a

train_domain,unieval_factuality,prometheus_factuality,prometheus_consistency,unieval_relevance,prometheus_relevance,rouge_rouge1,rouge_rougeLsum,moverscore,unieval_fluency,prometheus_fluency,unieval_coherence,prometheus_coherence,compression_sentence,compression_token,prometheus_coverage
Convo,0.74,2.29,2.38,0.69,2.63,0.26,0.17,0.05,0.85,2.73,0.73,2.64,0.83,0.72,-1.00
News+,0.80,-1.00,-1.00,0.73,-1.00,0.42,0.21,0.17,0.76,-1.00,0.81,-1.00,0.55,0.54,-1.00
Science,0.72,0.99,1.00,0.63,1.21,0.26,0.15,0.05,0.77,0.92,0.62,0.91,0.64,0.64,-1.00



,unieval_factuality,prometheus_factuality,prometheus_consistency,unieval_relevance,prometheus_relevance,rouge_rouge1,rouge_rougeLsum,moverscore,unieval_fluency,prometheus_fluency,unieval_coherence,prometheus_coherence,compression_sentence,compression_token,prometheus_coverage
train_domain,,,,,,,,,,,,,,,
Convo,0.74,2.29,2.38,0.69,2.63,0.26,0.17,0.05,0.85,2.73,0.73,2.64,0.83,0.72,-1.00
News+,0.80,-1.00,-1.00,0.73,-1.00,0.42,0.21,0.17,0.76,-1.00,0.81,-1.00,0.55,0.54,-1.00
Science,0.72,0.99,1.00,0.63,1.21,0.26,0.15,0.05,0.77,0.92,0.62,0.91,0.64,0.64,-1.00


# Q: Which train domain transfers the best to other domains?

Limit to models that are trained and tested on _different_ domains. Also, focus on performance _loss_ versus absolute performance. 


In [12]:
cross_performance = results_df[results_df.train_domain!=results_df.test_domain].groupby(["model_type", "train_domain", "test_domain"]).mean(numeric_only=True)
print(cross_performance.to_csv())
cross_performance

model_type,train_domain,test_domain,unieval_factuality,prometheus_factuality,prometheus_consistency,unieval_relevance,prometheus_relevance,rouge_rouge1,rouge_rougeLsum,moverscore,unieval_fluency,prometheus_fluency,unieval_coherence,prometheus_coherence,compression_sentence,compression_token,prometheus_coverage
ConvoSumm,Convo,News+,0.7591189466775365,-1.0,-1.0,0.5070198568057779,-1.0,0.2362862336067719,0.13624412376100595,0.004544066298236168,0.904482613918518,-1.0,0.578362022112479,-1.0,0.7661639042019806,0.8441804327242147,-1.0
ConvoSumm,Convo,Science,0.6820770616738082,1.9426617569700158,1.9200420831141505,0.634548276780649,2.2051551814834296,0.23137961259831868,0.14082646645653718,0.015661103842486895,0.9081399542147147,2.17,0.6100838214473558,2.054708048395581,0.7586574122367228,0.8454746007797356,-1.0
ConvoSumm,News+,Convo,0.6969226520006567,1.62,1.672,0.7576750387866421,2.084,0.22889761798768465,0.13203723709223633,0.007668224319735004,0.6989360899262449,1.788,0.7797604320135647

unieval_factuality  \
model_type         train_domain test_domain                       
ConvoSumm          Convo        News+                      0.76   
                                Science                    0.68   
                   News+        Convo                      0.70   
                                Science                    0.71   
                   Science      Convo                      0.46   
                                News+                      0.64   
Llama 3.1          Convo        News+                      0.88   
                                Science                    0.88   
                   News+        Convo                      0.82   
                                Science                    0.87   
                   Science      Convo                      0.83   
                                News+                      0.88   
Llama 3.1 (0-shot) zero-shot    Convo                      0.82   
                                News+                      0.88   
                                Science                    0.88   
PRIMERA            Convo        News+                      0.81   
                                Science                    0.83   
                   News+        Convo                      0.81   
                                Science                    0.82   
                   Science      Convo                      0.82   
                                News+                      0.81   
SummN              Convo        News+                      0.90   
                                Science                    0.80   
                   News+        Convo                      0.67   
                                Science                    0.64   
                   Science      Convo                      0.40   
                                News+                      0.64   

                                             prometheus_factuality  \
model_type         train_domain test_domain                          
ConvoSumm          Convo        News+                        -1.00   
                                Science                       1.94   
                   News+        Convo                         1.62   
                                Science                       1.82   
                   Science      Convo                         1.23   
                                News+                        -1.00   
Llama 3.1          Convo        News+                        -1.00   
                                Science                      -1.00   
                   News+        Convo                         3.06   
                                Science                      -1.00   
                   Science      Convo                         3.18   
                                News+                        -1.00   
Llama 3.1 (0-shot) zero-shot    Convo                         3.00   
                                News+                        -1.00   
                                Science                      -1.00   
PRIMERA            Convo        News+                        -1.00   
                                Science                       1.92   
                   News+        Convo                         1.45   
                                Science                       1.93   
                   Science      Convo                         1.38   
                                News+                        -1.00   
SummN              Convo        News+                        -1.00   
                                Science                       2.48   
                   News+        Convo                         1.50   
                                Science                       1.59   
                   Science      Convo                         1.13   
                                News+                        -1.00   

                                             prometheus_consistency  \

In [13]:
cross_performance.groupby(["train_domain", "test_domain"]).mean()


unieval_factuality  prometheus_factuality  \
train_domain test_domain                                              
Convo        News+                      0.84                  -1.00   
             Science                    0.80                   1.34   
News+        Convo                      0.75                   1.91   
             Science                    0.76                   1.08   
Science      Convo                      0.63                   1.73   
             News+                      0.74                  -1.00   
zero-shot    Convo                      0.82                   3.00   
             News+                      0.88                  -1.00   
             Science                    0.88                  -1.00   

                          prometheus_consistency  unieval_relevance  \
train_domain test_domain                                              
Convo        News+                         -1.00               0.62   
             Science                        1.33               0.71   
News+        Convo                          1.99               0.71   
             Science                        0.39               0.71   
Science      Convo                          1.86               0.47   
             News+                         -1.00               0.48   
zero-shot    Convo                          3.26               0.94   
             News+                         -1.00               0.92   
             Science                       -1.00               0.91   

                          prometheus_relevance  rouge_rouge1  rouge_rougeLsum  \
train_domain test_domain                                                        
Convo        News+                       -1.00          0.31             0.17   
             Science                      1.54          0.24             0.14   
News+        Convo                        2.37          0.21             0.13   
             Science                      1.42          0.24             0.14   
Science      Convo                        2.01          0.21             0.14   
             News+                       -1.00          0.34             0.18   
zero-shot    Convo                        3.43          0.22             0.15   
             News+                       -1.00          0.40             0.21   
             Science                     -1.00          0.25             0.16   

                          moverscore  unieval_fluency  prometheus_fluency  \
train_domain test_domain                                                    
Convo        News+              0.08             0.84               -1.00   
             Science            0.04             0.89                1.40   
News+        Convo              0.01             0.74                2.21   
             Science            0.05             0.83                0.38   
Science      Convo             -0.00             0.70                1.99   
             News+              0.09             0.71               -1.00   
zero-shot    Convo              0.05             0.92                4.04   
             News+              0.17             0.95               -1.00   
             Science            0.08             0.93               -1.00   

                          unieval_coherence  prometheus_coherence  \
train_domain test_domain                                            
Convo        News+                     0.76                 -1.00   
             Science                   0.80                  1.31   
News+        Convo                     0.80                  2.19   
             Science                   0.77                  0.38   
Science      Convo                     0.55                  1.93   
             News+                     0.57                 -1.00   
zero-shot    Convo                     0.96                  3.95   
             News+                     0.94                 -1.00   
             Science            

In [14]:
# Limit to models where the model is trained and tested on different domains
def safe_divide(num, denom):
    if denom == 0:
        return 0.0
    return num / denom

def get_relative_performance_loss(row):
    key = [row.train_domain, row.model_type]
    if row.train_domain == "zero-shot":
        key[0] = row.test_domain
        key[1] = "ICL"
    original_performance = same_domain_performance_dict.get(
        tuple(key),
        {}
    )
    rel_performance = {
        "train_domain": row.train_domain,
        "test_domain": row.test_domain,
        "model_type": row.model_type
    }
    rel_performance.update({
        m: safe_divide(
            row[m] - original_performance.get(m, 0.0), 
            original_performance.get(m, 0.0)
        ) for m in wanted_metrics
    })
    return rel_performance

performance_diff = results_df[results_df.train_domain!=results_df.test_domain].apply(
    get_relative_performance_loss,
    axis="columns",
    result_type="expand"
)


In [15]:
diff = performance_diff.groupby(["model_type", "train_domain", "test_domain"]).mean(numeric_only=True)

print(diff.to_csv())

diff


model_type,train_domain,test_domain,unieval_factuality,prometheus_factuality,prometheus_consistency,unieval_relevance,prometheus_relevance,rouge_rouge1,rouge_rougeLsum,moverscore,unieval_fluency,prometheus_fluency,unieval_coherence,prometheus_coherence,compression_sentence,compression_token,prometheus_coverage
ConvoSumm,Convo,News+,0.1771797064344579,-1.4528985507246377,-1.4448398576512456,-0.27383792063492,-1.4098360655737705,-0.23466650469936362,-0.35414408709721795,-0.9184079769375271,-0.011042566259500378,-1.374251497005988,-0.03548471768299426,-1.4019292604501608,-0.16148944502507964,-0.06848520524538919,0.0
ConvoSumm,Convo,Science,0.05770943900300757,-0.12017130572010164,-0.1458887530631004,-0.09118964486419427,-0.09624787644121736,-0.2505590996643519,-0.33242180621523865,-0.71879346337971,-0.007043645972937263,-0.18787425149700607,0.017416681564018967,-0.17415271366737095,-0.16970475340646188,-0.06705714953136825,0.0
ConvoSumm,News+,Convo,-0.10119112668158631,-2.62,-2.6719999999

unieval_factuality  \
model_type         train_domain test_domain                       
ConvoSumm          Convo        News+                      0.18   
                                Science                    0.06   
                   News+        Convo                     -0.10   
                                Science                   -0.09   
                   Science      Convo                     -0.23   
                                News+                      0.07   
Llama 3.1          Convo        News+                      0.06   
                                Science                    0.06   
                   News+        Convo                     -0.06   
                                Science                   -0.00   
                   Science      Convo                     -0.05   
                                News+                     -0.00   
Llama 3.1 (0-shot) zero-shot    Convo                      0.00   
                                News+                      0.00   
                                Science                    0.00   
PRIMERA            Convo        News+                     -0.00   
                                Science                    0.02   
                   News+        Convo                      0.00   
                                Science                    0.01   
                   Science      Convo                     -0.02   
                                News+                     -0.03   
SummN              Convo        News+                      0.34   
                                Science                    0.20   
                   News+        Convo                     -0.08   
                                Science                   -0.12   
                   Science      Convo                     -0.29   
                                News+                      0.14   

                                             prometheus_factuality  \
model_type         train_domain test_domain                          
ConvoSumm          Convo        News+                        -1.45   
                                Science                      -0.12   
                   News+        Convo                        -2.62   
                                Science                      -2.82   
                   Science      Convo                        -0.22   
                                News+                        -1.64   
Llama 3.1          Convo        News+                        -1.33   
                                Science                      -1.33   
                   News+        Convo                        -4.06   
                                Science                       0.00   
                   Science      Convo                        -4.18   
                                News+                         0.00   
Llama 3.1 (0-shot) zero-shot    Convo                         0.00   
                                News+                         0.00   
                                Science                       0.00   
PRIMERA            Convo        News+                        -1.69   
                                Science                       0.33   
                   News+        Convo                        -2.45   
                                Science                      -2.93   
                   Science      Convo                        -0.28   
                                News+                        -1.52   
SummN              Convo        News+                        -1.41   
                                Science                       0.01   
                   News+        Convo                        -2.50   
                                Science                      -2.59   
                   Science      Convo                        -0.23   
                                News+                        -1.68   

                                             prometheus_consistency  \

In [16]:
model_display_order = [
    "ConvoSumm",
    "PRIMERA",
    "SummN",
    "Llama 3.1",
    "Llama 3.1 (0-shot)"
]
diff_data = performance_diff.groupby(["model_type"]).mean(numeric_only=True).reindex(model_display_order)
print(diff_data.to_csv())
diff_data.T

model_type,unieval_factuality,prometheus_factuality,prometheus_consistency,unieval_relevance,prometheus_relevance,rouge_rouge1,rouge_rougeLsum,moverscore,unieval_fluency,prometheus_fluency,unieval_coherence,prometheus_coherence,compression_sentence,compression_token,prometheus_coverage
ConvoSumm,-0.020103900379910444,-1.4777550336285563,-1.008607945709916,-0.26438442609039964,-1.5912319272270778,-0.27382487271847006,-0.28059359733659134,-0.8606276415574721,-0.04747891661894078,-1.0225906507214306,-0.14005126299678236,-1.0356382740614045,0.07485414056002411,0.009979105952065068,0.0
PRIMERA,-0.002926628326172088,-1.423834841018475,-1.432059122505514,0.056339873780841064,-1.5372872781771625,0.12196202740336125,0.0636931059197022,-3.323094882928501,0.01871434969724588,-1.3874681345300928,0.016183034803128975,-1.3840491311767469,0.014051542799038039,0.023038807842389968,0.0
SummN,0.030923751215561412,-1.398180928847002,-1.398142394306632,-0.18810801070159588,-1.572368831985261,-0.3021009992

model_type,ConvoSumm,PRIMERA,SummN,Llama 3.1,Llama 3.1 (0-shot)
unieval_factuality,-0.02,-0.00,0.03,-0.00,0.00
prometheus_factuality,-1.48,-1.42,-1.40,-1.82,0.00
prometheus_consistency,-1.01,-1.43,-1.40,-1.87,0.00
unieval_relevance,-0.26,0.06,-0.19,0.00,0.00
prometheus_relevance,-1.59,-1.54,-1.57,-1.91,0.00
rouge_rouge1,-0.27,0.12,-0.30,0.10,0.00
rouge_rougeLsum,-0.28,0.06,-0.28,0.04,0.00
moverscore,-0.86,-3.32,-0.88,0.35,0.00
unieval_fluency,-0.05,0.02,0.00,-0.00,0.00
prometheus_fluency,-1.02,-1.39,-1.44,-2.09,0.00


In [17]:
performance_diff.groupby(["train_domain"]).mean(numeric_only=True)

,unieval_factuality,prometheus_factuality,prometheus_consistency,unieval_relevance,prometheus_relevance,rouge_rouge1,rouge_rougeLsum,moverscore,unieval_fluency,prometheus_fluency,unieval_coherence,prometheus_coherence,compression_sentence,compression_token,prometheus_coverage
train_domain,,,,,,,,,,,,,,,
Convo,0.11,-0.87,-0.88,-0.01,-0.85,0.13,-0.01,-2.38,0.02,-0.86,0.06,-0.87,-0.15,-0.02,0.00
News+,-0.05,-2.49,-2.19,0.00,-2.89,-0.46,-0.37,-0.80,0.04,-2.29,-0.03,-2.28,0.16,-0.00,0.00
Science,-0.05,-1.22,-1.21,-0.29,-1.21,0.06,0.03,-0.36,-0.09,-1.30,-0.13,-1.31,0.15,0.06,0.00
zero-shot,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


In [18]:
diff_data = performance_diff.groupby(["train_domain", "test_domain"]).mean(numeric_only=True)
print(diff_data.to_csv(float_format='%.2f'))
diff_data

train_domain,test_domain,unieval_factuality,prometheus_factuality,prometheus_consistency,unieval_relevance,prometheus_relevance,rouge_rouge1,rouge_rougeLsum,moverscore,unieval_fluency,prometheus_fluency,unieval_coherence,prometheus_coherence,compression_sentence,compression_token,prometheus_coverage
Convo,News+,0.14,-1.47,-1.45,-0.10,-1.40,0.31,0.09,-3.04,-0.02,-1.43,0.03,-1.45,-0.13,0.00,0.00
Convo,Science,0.08,-0.28,-0.31,0.08,-0.31,-0.04,-0.11,-1.72,0.06,-0.29,0.09,-0.29,-0.18,-0.05,0.00
News+,Convo,-0.06,-2.91,-3.00,-0.03,-3.37,-0.49,-0.39,-0.92,-0.02,-3.21,-0.01,-3.19,0.35,0.09,0.00
News+,Science,-0.05,-2.08,-1.39,0.04,-2.42,-0.43,-0.35,-0.68,0.11,-1.38,-0.05,-1.38,-0.03,-0.09,0.00
Science,Convo,-0.15,-1.23,-1.22,-0.30,-1.28,-0.19,-0.11,-1.28,-0.10,-1.36,-0.16,-1.38,0.26,0.08,0.00
Science,News+,0.04,-1.21,-1.21,-0.28,-1.14,0.32,0.17,0.57,-0.07,-1.23,-0.11,-1.24,0.05,0.04,0.00
zero-shot,Convo,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
zero-shot,News+

unieval_factuality  prometheus_factuality  \
train_domain test_domain                                              
Convo        News+                      0.14                  -1.47   
             Science                    0.08                  -0.28   
News+        Convo                     -0.06                  -2.91   
             Science                   -0.05                  -2.08   
Science      Convo                     -0.15                  -1.23   
             News+                      0.04                  -1.21   
zero-shot    Convo                      0.00                   0.00   
             News+                      0.00                   0.00   
             Science                    0.00                   0.00   

                          prometheus_consistency  unieval_relevance  \
train_domain test_domain                                              
Convo        News+                         -1.45              -0.10   
             Science                       -0.31               0.08   
News+        Convo                         -3.00              -0.03   
             Science                       -1.39               0.04   
Science      Convo                         -1.22              -0.30   
             News+                         -1.21              -0.28   
zero-shot    Convo                          0.00               0.00   
             News+                          0.00               0.00   
             Science                        0.00               0.00   

                          prometheus_relevance  rouge_rouge1  rouge_rougeLsum  \
train_domain test_domain                                                        
Convo        News+                       -1.40          0.31             0.09   
             Science                     -0.31         -0.04            -0.11   
News+        Convo                       -3.37         -0.49            -0.39   
             Science                     -2.42         -0.43            -0.35   
Science      Convo                       -1.28         -0.19            -0.11   
             News+                       -1.14          0.32             0.17   
zero-shot    Convo                        0.00          0.00             0.00   
             News+                        0.00          0.00             0.00   
             Science                      0.00          0.00             0.00   

                          moverscore  unieval_fluency  prometheus_fluency  \
train_domain test_domain                                                    
Convo        News+             -3.04            -0.02               -1.43   
             Science           -1.72             0.06               -0.29   
News+        Convo             -0.92            -0.02               -3.21   
             Science           -0.68             0.11               -1.38   
Science      Convo             -1.28            -0.10               -1.36   
             News+              0.57            -0.07               -1.23   
zero-shot    Convo              0.00             0.00                0.00   
             News+              0.00             0.00                0.00   
             Science            0.00             0.00                0.00   

                          unieval_coherence  prometheus_coherence  \
train_domain test_domain                                            
Convo        News+                     0.03                 -1.45   
             Science                   0.09                 -0.29   
News+        Convo                    -0.01                 -3.19   
             Science                  -0.05                 -1.38   
Science      Convo                    -0.16                 -1.38   
             News+                    -0.11                 -1.24   
zero-shot    Convo                     0.00                  0.00   
             News+                     0.00                  0.00   
             Science            

In [19]:
a = performance_diff.groupby(["train_domain"]).mean(numeric_only=True)
print(a.to_csv(float_format='%.2f'))
a

train_domain,unieval_factuality,prometheus_factuality,prometheus_consistency,unieval_relevance,prometheus_relevance,rouge_rouge1,rouge_rougeLsum,moverscore,unieval_fluency,prometheus_fluency,unieval_coherence,prometheus_coherence,compression_sentence,compression_token,prometheus_coverage
Convo,0.11,-0.87,-0.88,-0.01,-0.85,0.13,-0.01,-2.38,0.02,-0.86,0.06,-0.87,-0.15,-0.02,0.00
News+,-0.05,-2.49,-2.19,0.00,-2.89,-0.46,-0.37,-0.80,0.04,-2.29,-0.03,-2.28,0.16,-0.00,0.00
Science,-0.05,-1.22,-1.21,-0.29,-1.21,0.06,0.03,-0.36,-0.09,-1.30,-0.13,-1.31,0.15,0.06,0.00
zero-shot,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00



,unieval_factuality,prometheus_factuality,prometheus_consistency,unieval_relevance,prometheus_relevance,rouge_rouge1,rouge_rougeLsum,moverscore,unieval_fluency,prometheus_fluency,unieval_coherence,prometheus_coherence,compression_sentence,compression_token,prometheus_coverage
train_domain,,,,,,,,,,,,,,,
Convo,0.11,-0.87,-0.88,-0.01,-0.85,0.13,-0.01,-2.38,0.02,-0.86,0.06,-0.87,-0.15,-0.02,0.00
News+,-0.05,-2.49,-2.19,0.00,-2.89,-0.46,-0.37,-0.80,0.04,-2.29,-0.03,-2.28,0.16,-0.00,0.00
Science,-0.05,-1.22,-1.21,-0.29,-1.21,0.06,0.03,-0.36,-0.09,-1.30,-0.13,-1.31,0.15,0.06,0.00
zero-shot,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


In [20]:
a = performance_diff.groupby(["test_domain"]).mean(numeric_only=True)
print(a.to_csv(float_format='%.2f'))
a

test_domain,unieval_factuality,prometheus_factuality,prometheus_consistency,unieval_relevance,prometheus_relevance,rouge_rouge1,rouge_rougeLsum,moverscore,unieval_fluency,prometheus_fluency,unieval_coherence,prometheus_coherence,compression_sentence,compression_token,prometheus_coverage
Convo,-0.09,-1.84,-1.87,-0.15,-2.07,-0.30,-0.22,-0.98,-0.05,-2.03,-0.08,-2.03,0.27,0.07,0.00
News+,0.08,-1.19,-1.18,-0.17,-1.13,0.28,0.11,-1.10,-0.04,-1.18,-0.03,-1.19,-0.04,0.02,0.00
Science,0.01,-1.05,-0.75,0.05,-1.21,-0.21,-0.20,-1.07,0.07,-0.74,0.02,-0.75,-0.09,-0.06,0.00



,unieval_factuality,prometheus_factuality,prometheus_consistency,unieval_relevance,prometheus_relevance,rouge_rouge1,rouge_rougeLsum,moverscore,unieval_fluency,prometheus_fluency,unieval_coherence,prometheus_coherence,compression_sentence,compression_token,prometheus_coverage
test_domain,,,,,,,,,,,,,,,
Convo,-0.09,-1.84,-1.87,-0.15,-2.07,-0.30,-0.22,-0.98,-0.05,-2.03,-0.08,-2.03,0.27,0.07,0.00
News+,0.08,-1.19,-1.18,-0.17,-1.13,0.28,0.11,-1.10,-0.04,-1.18,-0.03,-1.19,-0.04,0.02,0.00
Science,0.01,-1.05,-0.75,0.05,-1.21,-0.21,-0.20,-1.07,0.07,-0.74,0.02,-0.75,-0.09,-0.06,0.00


# Q: Evaluating the metrics

Theoretically, the metrics should all be very high on the ground truth datasets. Is that the case?

In [21]:
a = results_df[results_df.model_type=="ground_truth"]

print(a.to_csv(float_format='%.2f'))
a

,model_type,train_domain,test_domain,unieval_factuality,prometheus_factuality,prometheus_consistency,unieval_relevance,prometheus_relevance,rouge_rouge1,rouge_rougeLsum,moverscore,unieval_fluency,prometheus_fluency,unieval_coherence,prometheus_coherence,compression_sentence,compression_token,prometheus_coverage
0,ground_truth,Convo,Convo,0.71,3.03,3.13,0.99,3.26,-1.00,-1.00,1.00,0.90,3.28,0.73,3.48,0.90,0.85,-1
14,ground_truth,Science,Science,0.55,1.82,1.82,0.99,2.76,-1.00,-1.00,1.00,0.66,1.56,0.43,1.49,0.74,0.75,-1
28,ground_truth,News+,News+,0.70,-1.00,-1.00,0.98,-1.00,-1.00,-1.00,1.00,0.76,-1.00,0.75,-1.00,0.41,0.41,-1



,model_type,train_domain,test_domain,unieval_factuality,prometheus_factuality,prometheus_consistency,unieval_relevance,prometheus_relevance,rouge_rouge1,rouge_rougeLsum,moverscore,unieval_fluency,prometheus_fluency,unieval_coherence,prometheus_coherence,compression_sentence,compression_token,prometheus_coverage
0,ground_truth,Convo,Convo,0.71,3.03,3.13,0.99,3.26,-1.00,-1.00,1.00,0.90,3.28,0.73,3.48,0.90,0.85,-1
14,ground_truth,Science,Science,0.55,1.82,1.82,0.99,2.76,-1.00,-1.00,1.00,0.66,1.56,0.43,1.49,0.74,0.75,-1
28,ground_truth,News+,News+,0.70,-1.00,-1.00,0.98,-1.00,-1.00,-1.00,1.00,0.76,-1.00,0.75,-1.00,0.41,0.41,-1


# Checking Domain Performance Shift

In [22]:
df = results_df.copy()
# --- 2. Isolate In-Domain Baselines ---
# In-domain is where the training and testing domain are the same.
in_domain_mask = df['train_domain'] == df['test_domain']
in_domain_baselines = df[in_domain_mask].set_index(['model_type', 'train_domain'])

# Define identifier and metric columns
id_cols = ['model_type', 'train_domain', 'test_domain']
metric_cols = [col for col in df.columns if col not in id_cols]

# --- 3. Calculate Performance Difference for OOD cases ---
# Out-of-domain (OOD) is where training and testing domains differ.
ood_df = df[~in_domain_mask].copy()

# Exclude zero-shot models as they don't have a traditional "in-domain" baseline
ood_df = ood_df[ood_df['train_domain'] != 'zero-shot']
print("Note: 'Llama 3.1 (0-shot)' is excluded as it has no in-domain baseline.\n")

results_list = []
for index, ood_row in ood_df.iterrows():
    model = ood_row['model_type']
    train_domain = ood_row['train_domain']
    
    try:
        # Find the corresponding in-domain baseline performance
        baseline_row = in_domain_baselines.loc[(model, train_domain)]
        
        # Create a dictionary for the results
        diff_row = {
            'model_type': model,
            'train_domain': train_domain,
            'test_domain': ood_row['test_domain']
        }
        
        # Calculate the difference for each metric
        for metric in metric_cols:
            diff = ood_row[metric] - baseline_row[metric]
            diff_row[metric] = diff
            
        results_list.append(diff_row)

    except KeyError:
        print(f"Warning: Could not find in-domain baseline for model='{model}', train_domain='{train_domain}'. Skipping.")


# --- 4. Create and Display the Final DataFrame ---
diff_df = pd.DataFrame(results_list)

print("--- Performance Difference (Out-of-Domain vs. In-Domain) ---")
# Using to_string() to display all columns without truncation
a = diff_df.set_index(["model_type", "train_domain", "test_domain"]).sort_index()
a.loc["PRIMERA"].T

Note: 'Llama 3.1 (0-shot)' is excluded as it has no in-domain baseline.

--- Performance Difference (Out-of-Domain vs. In-Domain) ---


train_domain           Convo         News+         Science      
test_domain            News+ Science Convo Science   Convo News+
unieval_factuality     -0.00    0.01  0.00    0.01   -0.02 -0.02
prometheus_factuality  -2.45    0.47  2.45    2.93   -0.55 -2.93
prometheus_consistency -2.58    0.40  2.52    2.96   -0.44 -2.97
unieval_relevance       0.02    0.21 -0.01    0.19   -0.21 -0.20
prometheus_relevance   -2.93    0.45  2.98    3.36   -0.43 -3.36
rouge_rouge1            0.18    0.05 -0.18   -0.14   -0.05  0.14
rouge_rougeLsum         0.07    0.02 -0.07   -0.05   -0.02  0.05
moverscore              0.14    0.07 -0.14   -0.07   -0.07  0.07
unieval_fluency        -0.04    0.14  0.04    0.18   -0.14 -0.18
prometheus_fluency     -2.34    0.49  2.29    2.83   -0.50 -2.82
unieval_coherence       0.01    0.05  0.03    0.04   -0.03 -0.04
prometheus_coherence   -2.30    0.50  2.27    2.80   -0.52 -2.79
compression_sentence   -0.12   -0.11  0.13    0.01    0.11 -0.01
compression_token      -0.07   -0.07  0.11    0.00    0.09 -0.01
prometheus_coverage     0.00    0.00  0.00    0.00    0.00  0.00

# Identifying why PRIMERA breaks the pattern

In [25]:
# --- 2. Isolate PRIMERA model results ---
primera_df = results_df[results_df['model_type'] == 'PRIMERA'].copy()

# --- 3. Define key metrics for analysis ---
metrics_to_analyze = [
    'unieval_relevance',
    'rouge_rougeLsum',
    "moverscore"
]

# --- 4. Group by training domain and analyze performance ---
for train_domain, group_df in primera_df.groupby('train_domain'):
    print(f"--- Analysis for PRIMERA trained on: {train_domain} ---")
    
    # Separate in-domain and out-of-domain (OOD) results
    in_domain_mask = group_df['test_domain'] == train_domain
    in_domain_row = group_df[in_domain_mask]
    ood_rows = group_df[~in_domain_mask]
    
    if in_domain_row.empty:
        print("  No in-domain results found for this training domain.\n")
        continue

    for metric in metrics_to_analyze:
        # Get the single in-domain score
        in_domain_score = in_domain_row[metric].iloc[0]
        
        # Get the series of out-of-domain scores
        ood_scores = ood_rows[metric]
        
        # Calculate the average of the OOD scores
        ood_average = ood_scores.mean()
        
        print(f"  Metric: {metric}")
        print(f"    In-Domain Score ({train_domain}): {in_domain_score:.2f}")
        
        # Display individual OOD scores
        ood_scores_list = [f"{score:.2f} ({test_dom})" for score, test_dom in zip(ood_scores, ood_rows['test_domain'])]
        print(f"    Out-of-Domain Scores: {', '.join(ood_scores_list)}")
        
        print(f"    Out-of-Domain Average: {ood_average:.2f}")
        print("-" * 20)
        
    print("\n" + "="*50 + "\n")


--- Analysis for PRIMERA trained on: Convo ---
  Metric: unieval_relevance
    In-Domain Score (Convo): 0.40
    Out-of-Domain Scores: 0.62 (Science), 0.42 (News+)
    Out-of-Domain Average: 0.52
--------------------
  Metric: rouge_rougeLsum
    In-Domain Score (Convo): 0.11
    Out-of-Domain Scores: 0.13 (Science), 0.18 (News+)
    Out-of-Domain Average: 0.16
--------------------
  Metric: moverscore
    In-Domain Score (Convo): -0.01
    Out-of-Domain Scores: 0.06 (Science), 0.13 (News+)
    Out-of-Domain Average: 0.09
--------------------


--- Analysis for PRIMERA trained on: News+ ---
  Metric: unieval_relevance
    In-Domain Score (News+): 0.42
    Out-of-Domain Scores: 0.41 (Convo), 0.62 (Science)
    Out-of-Domain Average: 0.52
--------------------
  Metric: rouge_rougeLsum
    In-Domain Score (News+): 0.18
    Out-of-Domain Scores: 0.11 (Convo), 0.13 (Science)
    Out-of-Domain Average: 0.12
--------------------
  Metric: moverscore
    In-Domain Score (News+): 0.13
    Out-o

In [26]:
results_df.groupby(["model_type", "test_domain", "train_domain"])[metrics_to_analyze].mean(numeric_only=True)


unieval_relevance  \
model_type         test_domain train_domain                      
ConvoSumm          Convo       Convo                      0.70   
                               News+                      0.76   
                               Science                    0.23   
                   News+       Convo                      0.51   
                               News+                      0.79   
                               Science                    0.24   
                   Science     Convo                      0.63   
                               News+                      0.69   
                               Science                    0.49   
Llama 3.1          Convo       Convo                      0.93   
                               News+                      0.93   
                               Science                    0.94   
                   News+       Convo                      0.92   
                               News+                      0.92   
                               Science                    0.93   
                   Science     Convo                      0.91   
                               News+                      0.91   
                               Science                    0.91   
Llama 3.1 (0-shot) Convo       zero-shot                  0.94   
                   News+       zero-shot                  0.92   
                   Science     zero-shot                  0.91   
PRIMERA            Convo       Convo                      0.40   
                               News+                      0.41   
                               Science                    0.41   
                   News+       Convo                      0.42   
                               News+                      0.42   
                               Science                    0.42   
                   Science     Convo                      0.62   
                               News+                      0.62   
                               Science                    0.62   
SummN              Convo       Convo                      0.72   
                               News+                      0.73   
                               Science                    0.31   
                   News+       Convo                      0.61   
                               News+                      0.77   
                               Science                    0.34   
                   Science     Convo                      0.67   
                               News+                      0.64   
                               Science                    0.49   
ground_truth       Convo       Convo                      0.99   
                   News+       News+                      0.98   
                   Science     Science                    0.99   

                                             rouge_rougeLsum  moverscore  
model_type         test_domain train_domain                               
ConvoSumm          Convo       Convo                    0.21        0.06  
                               News+                    0.13        0.01  
                               Science                  0.15       -0.02  
                   News+       Convo                    0.14        0.00  
                               News+                    0.23        0.20  
                               Science                  0.15        0.03  
                   Science     Convo                    0.14        0.02  
                               News+                    0.14        0.04  
                               Science                  0.16        0.04  
Llama 3.1          Convo       Convo                    0.15        0.05  
                               News+                    0.15        0.05  
                               Science                  0.15        0.05  
                   News+       Convo                    0.21        0.17  
               